# Telco Churn: Data Understanding

In [1]:
from pathlib import Path
import pandas as pd

file_name = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

# Look in the project root or the notebooks folder's parent.
possible_paths = [
    Path.cwd() / "data" / "raw" / file_name,
    Path.cwd().parent / "data" / "raw" / file_name,
]
data_path = next(path for path in possible_paths if path.exists())

df = pd.read_csv(data_path)

print("Loaded from:", data_path.resolve())
print("Rows and columns:", df.shape)
display(df.head())

Loaded from: D:\telco-churn-analytics\data\raw\WA_Fn-UseC_-Telco-Customer-Churn.csv
Rows and columns: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
df.info()

print(
    "Blank TotalCharges cells:",
    df["TotalCharges"].astype(str).str.strip().eq("").sum()
)

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [3]:
print("Duplicate customer IDs:", df["customerID"].duplicated().sum())

for column in ["Churn", "Contract", "InternetService", "PaymentMethod", "SeniorCitizen"]:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False))

Duplicate customer IDs: 0

Churn
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Contract
Contract
Month-to-month    3875
Two year          1695
One year          1473
Name: count, dtype: int64

InternetService
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

PaymentMethod
PaymentMethod
Electronic check             2365
Mailed check                 1612
Bank transfer (automatic)    1544
Credit card (automatic)      1522
Name: count, dtype: int64

SeniorCitizen
SeniorCitizen
0    5901
1    1142
Name: count, dtype: int64


In [4]:
churned_customers = (df["Churn"] == "Yes").sum()
total_customers = len(df)
churn_rate = churned_customers / total_customers

print(f"Churned customers: {churned_customers:,}")
print(f"Total customers: {total_customers:,}")
print(f"Overall churn rate: {churn_rate:.1%}")

Churned customers: 1,869
Total customers: 7,043
Overall churn rate: 26.5%


In [5]:
blank_charges = df["TotalCharges"].astype(str).str.strip().eq("")

print("Blank TotalCharges:", blank_charges.sum())
print("Blank charges with tenure > 0:", (blank_charges & (df["tenure"] > 0)).sum())
display(df.loc[blank_charges, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]])

Blank TotalCharges: 11
Blank charges with tenure > 0: 0


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


In [6]:
df_clean = df.copy()

# Convert TotalCharges from text to numbers.
charges_text = df_clean["TotalCharges"].astype("string").str.strip()
blank_charges = charges_text.eq("")
charges_numeric = pd.to_numeric(charges_text.mask(blank_charges), errors="coerce")

# Stop if there are unexpected text values or blanks for non-new customers.
unexpected_invalid = (~blank_charges) & charges_numeric.isna()
assert unexpected_invalid.sum() == 0, "Unexpected non-numeric TotalCharges found"
assert df_clean.loc[blank_charges, "tenure"].eq(0).all(), (
    "A blank TotalCharges value has positive tenure; review before filling"
)

# Fill the 11 confirmed zero-tenure blanks with zero.
df_clean["TotalCharges"] = charges_numeric.fillna(0)

# Keep the source 0/1 field and add a readable label for analysis.
df_clean["SeniorCitizenLabel"] = df_clean["SeniorCitizen"].map({0: "No", 1: "Yes"})

print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)
print("Missing TotalCharges after cleaning:", df_clean["TotalCharges"].isna().sum())
print("TotalCharges type after cleaning:", df_clean["TotalCharges"].dtype)
display(df_clean.loc[blank_charges, ["tenure", "TotalCharges", "SeniorCitizen", "SeniorCitizenLabel"]])

Original shape: (7043, 21)
Cleaned shape: (7043, 22)
Missing TotalCharges after cleaning: 0
TotalCharges type after cleaning: Float64


,tenure,TotalCharges,SeniorCitizen,SeniorCitizenLabel
488,0,0.0,0,No
753,0,0.0,0,No
936,0,0.0,0,No
1082,0,0.0,0,No
1340,0,0.0,0,No
3331,0,0.0,0,No
3826,0,0.0,0,No
4380,0,0.0,0,No
5218,0,0.0,0,No
6670,0,0.0,0,No


In [7]:
clean_path = data_path.parents[2] / "data" / "clean" / "telco_clean.csv"

df_clean.to_csv(clean_path, index=False)

# Read the saved file back to confirm the export worked.
df_check = pd.read_csv(clean_path)

print("Saved to:", clean_path.resolve())
print("Saved shape:", df_check.shape)
print("Missing TotalCharges:", df_check["TotalCharges"].isna().sum())

assert df_check.shape == df_clean.shape
assert df_check["TotalCharges"].isna().sum() == 0
assert df_check["customerID"].is_unique

Saved to: D:\telco-churn-analytics\data\clean\telco_clean.csv
Saved shape: (7043, 22)
Missing TotalCharges: 0
